### **UNIVERSIDADE FEDERAL DE ITAJUBÁ**
### **Instituto de Matemática e Computação**
### **XMCO09 - Ciência de Redes** — Prof. Rafael Frinhani

## **Análise da evasão acadêmica por meio de redes de similaridade de atributos**

---

**Objetivo:** construir redes de similaridade entre estudantes (dataset de Realinho et al., 2022) seguindo os modelos de Porto & Frinhani (2021) e Porto (2022), detectar comunidades e relacioná-las à evasão, comparar a rede com modelos clássicos (AT3) e avaliar sua robustez (AT4).

| Seção | Conteúdo | CRISP-NET | Itens do formulário |
|---|---|---|---|
| 0 | Bibliotecas, parâmetros e funções auxiliares | — | — |
| 1 | Entendimento e preparação dos dados | Fases 2 e 3 | AT2.02–2.09 + EDA |
| 2 | Construção das redes (M1, M2, M3) e escolha da baseline | Fase 3 | AT2.04, 2.05, 2.08, 2.10 |
| 3 | Métricas, centralidades, visualização e comunidades (H1) | Fase 4 | AT2.10–2.13, AT3.01, H1 |
| 4 | Modelos de referência (ER, WS, BA) | Fase 4 | AT3.01–3.12 |
| 5 | Robustez (falhas × ataques) | Fase 4 | AT4.02–4.12 |

Cada célula indica entre parênteses o item do formulário de avaliação que ela atende. Itens apenas textuais (contexto, justificativas, limitações) são tratados no artigo.

---

## **0. Bibliotecas, Parâmetros e Funções Auxiliares**

**Q0.1** Instalação dos pacotes (executar apenas uma vez; ver `requirements.txt`).

In [1]:
# Descomente na primeira execução
# %pip install igraph pandas numpy scipy matplotlib

**Q0.2** Importação das bibliotecas e parâmetros globais.

In [2]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import igraph as ig
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Diretórios do projeto
DIR_DADOS = "dados"          # data.csv deve estar aqui
DIR_FIG = "figuras"          # figuras usadas no artigo
DIR_RES = "resultados"       # tabelas e dados intermediários
for d in (DIR_FIG, DIR_RES):
    os.makedirs(d, exist_ok=True)

# Estilo dos gráficos
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES_TARGET = {"Dropout": "#d62728", "Enrolled": "#ff7f0e", "Graduate": "#2ca02c"}

print(f"igraph {ig.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

igraph 1.0.0 | pandas 3.0.5 | numpy 2.5.3


**Q0.3** Funções auxiliares (as demais serão adicionadas nas seções seguintes).

In [3]:
def salvar_fig(nome):
    """Salva a figura atual em PNG (300 dpi) na pasta de figuras."""
    caminho = os.path.join(DIR_FIG, nome + ".png")
    plt.savefig(caminho, dpi=300, bbox_inches="tight")
    print(f"Figura salva: {caminho}")


def dominancia(serie):
    """Fração de registros com o valor mais frequente do atributo."""
    return serie.value_counts(normalize=True).iloc[0]


def cramer_v(x, y):
    """V de Cramér entre duas variáveis categóricas (0 = sem associação, 1 = total)."""
    tabela = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabela, correction=False)[0]
    n = tabela.values.sum()
    r, k = tabela.shape
    return math.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0


def discretizar_ordinal(serie, n=10):
    """Discretiza um atributo numérico em até n classes ordenadas (1..A), seguindo Porto (2022).
    - até n valores distintos: posto (rank) do valor;
    - mais de 50% no valor mínimo (ex.: 0 UC creditadas): classe 1 = mínimo, quantis no restante;
    - demais casos: n classes por quantis."""
    minimo = serie.min()
    if serie.nunique() <= n:
        return serie.rank(method="dense").astype(int)
    if (serie == minimo).mean() > 0.5:
        s = pd.Series(1, index=serie.index)
        pos = serie > minimo
        s[pos] = pd.qcut(serie[pos], q=n - 1, labels=False, duplicates="drop") + 2
        return s.astype(int)
    return (pd.qcut(serie, q=n, labels=False, duplicates="drop") + 1).astype(int)


def contar_outliers_iqr(serie):
    """Quantidade de valores fora de [Q1 - 1,5 IQR, Q3 + 1,5 IQR]."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)).sum())

print("Funções auxiliares carregadas.")

Funções auxiliares carregadas.


---
## **1. Entendimento e Preparação dos Dados** (CRISP-NET Fases 2 e 3)

**Q1.1** Leitura do *dataset*. O arquivo usa `;` como separador e possui BOM, e uma coluna tem `\t` no nome; os nomes são normalizados e traduzidos para facilitar a leitura.

In [4]:
dfBruto = pd.read_csv(os.path.join(DIR_DADOS, "data.csv"), sep=";", encoding="utf-8-sig")
dfBruto.columns = dfBruto.columns.str.strip()

mapaNomes = {
    "Marital status": "estado_civil", "Application mode": "modo_candidatura",
    "Application order": "ordem_candidatura", "Course": "curso",
    "Daytime/evening attendance": "turno", "Previous qualification": "qualif_previa",
    "Previous qualification (grade)": "nota_qualif_previa", "Nacionality": "nacionalidade",
    "Mother's qualification": "qualif_mae", "Father's qualification": "qualif_pai",
    "Mother's occupation": "ocup_mae", "Father's occupation": "ocup_pai",
    "Admission grade": "nota_admissao", "Displaced": "deslocado",
    "Educational special needs": "necess_especiais", "Debtor": "devedor",
    "Tuition fees up to date": "mensalidade_em_dia", "Gender": "genero",
    "Scholarship holder": "bolsista", "Age at enrollment": "idade_ingresso",
    "International": "internacional",
    "Curricular units 1st sem (credited)": "uc1_creditadas",
    "Curricular units 1st sem (enrolled)": "uc1_inscritas",
    "Curricular units 1st sem (evaluations)": "uc1_avaliacoes",
    "Curricular units 1st sem (approved)": "uc1_aprovadas",
    "Curricular units 1st sem (grade)": "uc1_nota",
    "Curricular units 1st sem (without evaluations)": "uc1_sem_avaliacao",
    "Curricular units 2nd sem (credited)": "uc2_creditadas",
    "Curricular units 2nd sem (enrolled)": "uc2_inscritas",
    "Curricular units 2nd sem (evaluations)": "uc2_avaliacoes",
    "Curricular units 2nd sem (approved)": "uc2_aprovadas",
    "Curricular units 2nd sem (grade)": "uc2_nota",
    "Curricular units 2nd sem (without evaluations)": "uc2_sem_avaliacao",
    "Unemployment rate": "taxa_desemprego", "Inflation rate": "taxa_inflacao",
    "GDP": "pib", "Target": "target",
}
df = dfBruto.rename(columns=mapaNomes)
assert set(df.columns) == set(mapaNomes.values()), "Colunas inesperadas no arquivo"

print(f"Registros (nós potenciais) = {df.shape[0]}")
print(f"Atributos = {df.shape[1] - 1} + target")
df.head()

Registros (nós potenciais) = 4424
Atributos = 36 + target


,estado_civil,modo_candidatura,ordem_candidatura,curso,turno,qualif_previa,nota_qualif_previa,nacionalidade,qualif_mae,qualif_pai,...,uc2_creditadas,uc2_inscritas,uc2_avaliacoes,uc2_aprovadas,uc2_nota,uc2_sem_avaliacao,taxa_desemprego,taxa_inflacao,pib,target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


**Q1.2** *(AT2.06)* Metadados dos atributos: grupo temático (usado no Modelo 2 – Multirelacional) e tipo (usado no Modelo 3 – Ordinal).

| Grupo | Significado | Equivalente em Porto (2022) |
|---|---|---|
| DM | Demográfico | Moradia e Mobilidade |
| SE | Socioeconômico | Condição Familiar / Renda e Despesas |
| AC | Acadêmico | — |
| MA | Macroeconômico (contexto do ano) | — |

In [5]:
grupo = {
    "estado_civil": "DM", "turno": "DM", "deslocado": "DM", "genero": "DM",
    "idade_ingresso": "DM", "nacionalidade": "DM", "internacional": "DM",
    "necess_especiais": "DM",
    "qualif_mae": "SE", "qualif_pai": "SE", "ocup_mae": "SE", "ocup_pai": "SE",
    "devedor": "SE", "mensalidade_em_dia": "SE", "bolsista": "SE",
    "modo_candidatura": "AC", "ordem_candidatura": "AC", "curso": "AC",
    "qualif_previa": "AC", "nota_qualif_previa": "AC", "nota_admissao": "AC",
    "taxa_desemprego": "MA", "taxa_inflacao": "MA", "pib": "MA",
}
for s in ("uc1", "uc2"):
    for v in ("creditadas", "inscritas", "avaliacoes", "aprovadas", "nota", "sem_avaliacao"):
        grupo[f"{s}_{v}"] = "AC"

nominais = ["estado_civil", "modo_candidatura", "curso", "qualif_previa", "qualif_mae",
            "qualif_pai", "ocup_mae", "ocup_pai", "nacionalidade"]
binarios = ["turno", "deslocado", "necess_especiais", "devedor", "mensalidade_em_dia",
            "genero", "bolsista", "internacional"]
tipo = {c: ("nominal" if c in nominais else "binario" if c in binarios else "ordinal")
        for c in grupo}

atributos = list(grupo.keys())
print(f"Total de atributos = {len(atributos)}")
print(pd.Series(grupo).value_counts().rename("qtd").to_frame().T)
print(pd.Series(tipo).value_counts().rename("qtd").to_frame().T)

Total de atributos = 36
     AC  DM  SE  MA
qtd  18   8   7   3
     ordinal  nominal  binario
qtd       19        9        8


**Q1.3** *(AT2.07)* Inspeção estrutural: tipos, valores ausentes e registros duplicados.

In [6]:
qtdAusentes = int(df.isna().sum().sum())
qtdDuplicados = int(df.duplicated().sum())
qtdDuplicadosAtrib = int(df[atributos].duplicated().sum())  # perfis idênticos (ignorando target)

print(f"Valores ausentes = {qtdAusentes}")
print(f"Registros duplicados = {qtdDuplicados}")
print(f"Perfis idênticos (sem target) = {qtdDuplicadosAtrib}")
print(f"Tipos de dados: {df.dtypes.value_counts().to_dict()}")

Valores ausentes = 0
Registros duplicados = 0
Perfis idênticos (sem target) = 0
Tipos de dados: {dtype('int64'): 29, dtype('float64'): 7, <StringDtype(storage='python', na_value=nan)>: 1}


**Q1.4** *(AT2.02)* Distribuição da variável alvo (*Target*). Ela **não** entra no cálculo de similaridade; é usada apenas como rótulo dos nós (evita circularidade na H1).

In [7]:
contagemTarget = df["target"].value_counts()
propTarget = df["target"].value_counts(normalize=True)
for t in contagemTarget.index:
    print(f"{t:9s} = {contagemTarget[t]:5d} ({propTarget[t]*100:.1f}%)")

plt.figure(figsize=(5, 3.2))
plt.bar(contagemTarget.index, contagemTarget.values,
        color=[CORES_TARGET[t] for t in contagemTarget.index])
for i, v in enumerate(contagemTarget.values):
    plt.text(i, v + 30, f"{v} ({propTarget.iloc[i]*100:.1f}%)", ha="center", fontsize=9)
plt.ylabel("Estudantes")
plt.title("Distribuição do Target")
salvar_fig("1_4_target")
plt.show()

Graduate  =  2209 (49.9%)
Dropout   =  1421 (32.1%)
Enrolled  =   794 (17.9%)


Figura salva: figuras/1_4_target.png


**Q1.5** *(EDA – enunciado AT2)* Medidas descritivas dos atributos numéricos e contagem de *outliers* (IQR).

In [8]:
numericos = [c for c in atributos if tipo[c] == "ordinal"]
descricao = df[numericos].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
descricao["outliers_iqr"] = [contar_outliers_iqr(df[c]) for c in numericos]
descricao.round(2)

,mean,std,min,25%,50%,75%,max,outliers_iqr
idade_ingresso,23.27,7.59,17.00,19.00,20.00,25.00,70.00,441
ordem_candidatura,1.73,1.31,0.00,1.00,1.00,2.00,9.00,541
nota_qualif_previa,132.61,13.19,95.00,125.00,133.10,140.00,190.00,179
nota_admissao,126.98,14.48,95.00,117.90,126.10,134.80,190.00,86
taxa_desemprego,11.57,2.66,7.60,9.40,11.10,13.90,16.20,0
taxa_inflacao,1.23,1.38,-0.80,0.30,1.40,2.60,3.70,0
pib,0.00,2.27,-4.06,-1.70,0.32,1.79,3.51,0
uc1_creditadas,0.71,2.36,0.00,0.00,0.00,0.00,20.00,577
uc1_inscritas,6.27,2.48,0.00,5.00,6.00,7.00,26.00,424
uc1_avaliacoes,8.30,4.18,0.00,6.00,8.00,10.00,45.00,158


**Q1.6** *(EDA – enunciado AT2)* Distribuições de atributos selecionados (histogramas) e comparação por *Target* (boxplots).

In [9]:
destaque = ["idade_ingresso", "nota_admissao", "nota_qualif_previa",
            "uc1_aprovadas", "uc1_nota", "uc2_nota"]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), destaque):
    ax.hist(df[c], bins=30, color="#4c72b0", edgecolor="white")
    ax.set_title(c)
fig.suptitle("Distribuição de atributos numéricos", fontsize=13)
plt.tight_layout()
salvar_fig("1_6a_histogramas")
plt.show()

ordemT = ["Dropout", "Enrolled", "Graduate"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), destaque):
    bp = ax.boxplot([df.loc[df["target"] == t, c] for t in ordemT],
                    tick_labels=ordemT, patch_artist=True, showfliers=False)
    for patch, t in zip(bp["boxes"], ordemT):
        patch.set_facecolor(CORES_TARGET[t]); patch.set_alpha(0.6)
    ax.set_title(c)
fig.suptitle("Atributos numéricos por Target (sem outliers)", fontsize=13)
plt.tight_layout()
salvar_fig("1_6b_boxplots_target")
plt.show()

Figura salva: figuras/1_6a_histogramas.png


Figura salva: figuras/1_6b_boxplots_target.png


**Q1.7** *(EDA – enunciado AT2)* Matriz de correlação (Spearman) entre os atributos. Correlações muito altas indicam redundância (ex.: unidades curriculares do 1º e 2º semestres).

In [10]:
corr = df[atributos].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(atributos))); ax.set_xticklabels(atributos, rotation=90, fontsize=7)
ax.set_yticks(range(len(atributos))); ax.set_yticklabels(atributos, fontsize=7)
ax.grid(False)
plt.colorbar(im, fraction=0.046, pad=0.04, label="ρ de Spearman")
plt.title("Correlação entre atributos")
salvar_fig("1_7_correlacao")
plt.show()

# Pares mais correlacionados
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().abs().sort_values(ascending=False))
print("Pares com |ρ| > 0,8:")
print(pares[pares > 0.8].round(3).to_string())

Figura salva: figuras/1_7_correlacao.png
Pares com |ρ| > 0,8:
nacionalidade   internacional     1.000
uc1_inscritas   uc2_inscritas     0.962
uc1_creditadas  uc2_creditadas    0.914
uc1_aprovadas   uc2_aprovadas     0.892


**Q1.8** *(AT2.07, AT2.09)* Variabilidade dos atributos. Seguindo Porto (2022), atributos em que quase todos os registros têm o mesmo valor (**dominância ≥ 96%**) não ajudam a discriminar os estudantes e são removidos do modelo.

In [11]:
LIMIAR_DOMINANCIA = 0.96

tabDom = pd.DataFrame({
    "grupo": [grupo[c] for c in atributos],
    "tipo": [tipo[c] for c in atributos],
    "n_valores": [df[c].nunique() for c in atributos],
    "dominancia": [dominancia(df[c]) for c in atributos],
}, index=atributos).sort_values("dominancia", ascending=False)

plt.figure(figsize=(11, 4))
cores = ["#d62728" if d >= LIMIAR_DOMINANCIA else "#4c72b0" for d in tabDom["dominancia"]]
plt.bar(tabDom.index, tabDom["dominancia"], color=cores)
plt.axhline(LIMIAR_DOMINANCIA, color="black", ls="--", lw=1, label="limiar 96%")
plt.xticks(rotation=90, fontsize=8)
plt.ylabel("Fração do valor mais frequente")
plt.title("Dominância por atributo (vermelho = removido)")
plt.legend()
salvar_fig("1_8_dominancia")
plt.show()

tabDom.head(8).round(3)

Figura salva: figuras/1_8_dominancia.png


,grupo,tipo,n_valores,dominancia
necess_especiais,DM,binario,2,0.988
nacionalidade,DM,nominal,21,0.975
internacional,DM,binario,2,0.975
uc2_sem_avaliacao,AC,ordinal,10,0.936
uc1_sem_avaliacao,AC,ordinal,11,0.934
turno,DM,binario,2,0.891
devedor,SE,binario,2,0.886
estado_civil,DM,nominal,6,0.886


**Q1.9** *(AT2.03, AT2.07, AT2.08)* Relatório de qualidade dos dados (problema, quantidade, impacto e tratamento).

In [12]:
qtdZeroUC1 = int((df["uc1_inscritas"] == 0).sum())
qtdZeroUC2 = int((df["uc2_inscritas"] == 0).sum())
removDom = tabDom.index[tabDom["dominancia"] >= LIMIAR_DOMINANCIA].tolist()
removMacro = [c for c in atributos if grupo[c] == "MA"]
N = len(df)
# registros com pelo menos uma categoria rara (< 2%) em algum atributo nominal
mascRaro = pd.Series(False, index=df.index)
for c in nominais:
    freq = df[c].value_counts(normalize=True)
    mascRaro |= df[c].isin(freq[freq < 0.02].index)
qtdRaros = int(mascRaro.sum())

relatorio = pd.DataFrame([
    ["Valores ausentes", qtdAusentes, "—", "Nenhum tratamento necessário"],
    ["Registros duplicados", qtdDuplicados, "—", "Nenhum tratamento necessário"],
    ["Perfis idênticos (sem target)", qtdDuplicadosAtrib,
     "Pares com similaridade máxima", "Mantidos: são estudantes distintos"],
    ["Atributos com dominância ≥ 96%", len(removDom),
     "Não discriminam perfis; inflam similaridade", f"Removidos: {', '.join(removDom)}"],
    ["Atributos macroeconômicos", len(removMacro),
     "Iguais para toda a coorte: similaridade artificial", f"Removidos: {', '.join(removMacro)}"],
    ["UC 1º sem. inscritas = 0", qtdZeroUC1, "Nota 0 é estrutural, não desempenho",
     "Mantidos; discretização isola a classe 0"],
    ["UC 2º sem. inscritas = 0", qtdZeroUC2, "Idem", "Idem"],
    ["Outliers em idade (IQR)", contar_outliers_iqr(df["idade_ingresso"]),
     "Distorcem distâncias numéricas", "Discretização por quantis"],
    ["Registros com categoria rara (< 2%)", qtdRaros,
     "Quase nunca coincidem: reduzem similaridade", "Aglutinadas em 'outros'"],
    ["Classe Enrolled (desfecho em aberto)", int(contagemTarget["Enrolled"]),
     "Desfecho censurado", "Mantida e analisada separadamente"],
], columns=["problema", "qtd", "impacto", "tratamento"])
# proporção em relação aos registros (não se aplica às linhas sobre atributos)
relatorio["proporcao_%"] = [round(100 * q / N, 2) if pd.notna(q) and "Atributos" not in p else None
                           for p, q in zip(relatorio["problema"], relatorio["qtd"])]
relatorio["qtd"] = relatorio["qtd"].astype("Int64")
relatorio

,problema,qtd,impacto,tratamento,proporcao_%
0,Valores ausentes,0,—,Nenhum tratamento necessário,0.00
1,Registros duplicados,0,—,Nenhum tratamento necessário,0.00
2,Perfis idênticos (sem target),0,Pares com similaridade máxima,Mantidos: são estudantes distintos,0.00
3,Atributos com dominância ≥ 96%,3,Não discriminam perfis; inflam similaridade,"Removidos: necess_especiais, nacionalidade, in...",NaN
4,Atributos macroeconômicos,3,Iguais para toda a coorte: similaridade artifi...,"Removidos: taxa_desemprego, taxa_inflacao, pib",NaN
5,UC 1º sem. inscritas = 0,180,"Nota 0 é estrutural, não desempenho",Mantidos; discretização isola a classe 0,4.07
6,UC 2º sem. inscritas = 0,180,Idem,Idem,4.07
7,Outliers em idade (IQR),441,Distorcem distâncias numéricas,Discretização por quantis,9.97
8,Registros com categoria rara (< 2%),1125,Quase nunca coincidem: reduzem similaridade,Aglutinadas em 'outros',25.43
9,Classe Enrolled (desfecho em aberto),794,Desfecho censurado,Mantida e analisada separadamente,17.95


**Q1.10** *(AT2.09)* Seleção de atributos (Fase 3 – Preparação): remoção dos atributos de dominância ≥ 96% e dos macroeconômicos.

In [13]:
atribSel = [c for c in atributos if c not in removDom and c not in removMacro]

print(f"Atributos originais = {len(atributos)}")
print(f"Removidos (dominância) = {removDom}")
print(f"Removidos (macroeconômicos) = {removMacro}")
print(f"Atributos selecionados = {len(atribSel)}")
print(pd.Series({c: grupo[c] for c in atribSel}).value_counts().to_dict())

Atributos originais = 36
Removidos (dominância) = ['necess_especiais', 'nacionalidade', 'internacional']
Removidos (macroeconômicos) = ['taxa_desemprego', 'taxa_inflacao', 'pib']
Atributos selecionados = 30
{'AC': 18, 'SE': 7, 'DM': 5}


**Q1.11** *(AT2.05, AT2.09)* Transformação dos atributos (seguindo Porto, 2022):
* **Ordinais numéricos** com mais de 10 valores → **10 classes por quantis** (`qcut`); com até 10 valores → posto (*rank*) do valor. Atributos com mais de 50% dos registros no valor mínimo (ex.: UC creditadas) recebem uma classe própria para o mínimo e quantis para o restante.
* **Nominais** → categorias com frequência < 2% aglutinadas em `outros` e recodificadas.
* **Binários** → mantidos (0/1).

Resultado: todos os atributos passam a ser categorias inteiras `1..A_a`, prontas para os três modelos.

In [14]:
LIMIAR_RARO = 0.02
N_CLASSES = 10

def transformar(serie, t):
    """Retorna a série recodificada em inteiros 1..A e o número de estados A."""
    if t == "ordinal":
        s = discretizar_ordinal(serie, N_CLASSES)
    elif t == "nominal":
        freq = serie.value_counts(normalize=True)
        raros = freq[freq < LIMIAR_RARO].index
        s = serie.where(~serie.isin(raros), -1)            # -1 = 'outros'
        s = pd.Series(pd.factorize(s, sort=True)[0] + 1, index=serie.index)
    else:  # binario
        s = serie.astype(int) + 1
    return s.astype(int), int(s.nunique())

dfPrep = pd.DataFrame(index=df.index)
estados = {}
for c in atribSel:
    dfPrep[c], estados[c] = transformar(df[c], tipo[c])

resumoPrep = pd.DataFrame({
    "grupo": [grupo[c] for c in atribSel],
    "tipo": [tipo[c] for c in atribSel],
    "valores_orig": [df[c].nunique() for c in atribSel],
    "estados_final": [estados[c] for c in atribSel],
}, index=atribSel)
resumoPrep

,grupo,tipo,valores_orig,estados_final
estado_civil,DM,nominal,6,4
turno,DM,binario,2,2
deslocado,DM,binario,2,2
genero,DM,binario,2,2
idade_ingresso,DM,ordinal,46,8
qualif_mae,SE,nominal,29,7
qualif_pai,SE,nominal,34,7
ocup_mae,SE,nominal,32,10
ocup_pai,SE,nominal,46,12
devedor,SE,binario,2,2


**Q1.12** *(AT2.06)* Dicionário de dados final e exportação para as próximas seções (e para o GitHub).

In [15]:
dicionario = resumoPrep.copy()
dicionario["min_orig"] = [df[c].min() for c in atribSel]
dicionario["max_orig"] = [df[c].max() for c in atribSel]
dicionario["dominancia"] = [round(dominancia(df[c]), 3) for c in atribSel]
dicionario["aplica_a"] = "nó"
dicionario.index.name = "atributo"

dfPrep.insert(0, "id", range(len(dfPrep)))
dfPrep["target"] = df["target"].values     # apenas rótulo, fora da similaridade

dfPrep.to_csv(os.path.join(DIR_RES, "dados_preparados.csv"), index=False)
dicionario.to_csv(os.path.join(DIR_RES, "dicionario_dados.csv"))
relatorio.to_csv(os.path.join(DIR_RES, "relatorio_qualidade.csv"), index=False)

print(f"dfPrep: {dfPrep.shape[0]} nós × {len(atribSel)} atributos (+ id, target)")
print("Arquivos salvos em", DIR_RES)

dfPrep: 4424 nós × 30 atributos (+ id, target)
Arquivos salvos em resultados


---
## **2. Construção das Redes de Similaridade e Escolha da Baseline** (CRISP-NET Fase 3 / AT2)

Seguindo Porto & Frinhani (2021) e Porto (2022), os **nós** são os estudantes e as **arestas** ligam pares cuja similaridade de atributos é **maior ou igual a um limiar W**. A aresta é **não direcionada** (a similaridade é simétrica) e **ponderada** (peso = similaridade, em [0, 1]). Não há interação direta entre os estudantes: a rede é uma **aproximação** baseada em perfis.

Três modelos de similaridade são avaliados:

| Modelo | Similaridade entre os estudantes *x* e *y* | Origem |
|---|---|---|
| **M1** – Contagem Direta | $R_{xy} = \frac{1}{n}\sum_{a=1}^{n} [x_a = y_a]$ | Porto (2022), Modelo 1 |
| **M2** – Multirelacional | $R_{xy} = \sum_{G} w_G \frac{\sum_{a \in G}[x_a = y_a]}{P_{G,x} + P_{G,y}}$, normalizado por $\max R$ | Porto (2022), Modelo 2 (Menezes, 2012) |
| **M3** – Medida Ordinal | $R_{xy} = 1 - \sqrt{\frac{1}{n}\sum_a d_a(x,y)^2}$ | Porto (2022), Modelo 3 (Han et al., 2012) |

* **M2:** $P_{G,x}$ é o número de coincidências que *x* tem com **todos** os demais estudantes nos atributos do grupo *G* (relativiza a força da relação). Pesos **a priori** da literatura: **DM = 1, SE = 2, AC = 3** (17/33/50%, mesma escala de Porto), com base em Kim et al. (2023) e Jiménez-Macías et al. (2023). 
* **M3:** ordinais e binários são normalizados por $z = \frac{r-1}{A-1}$ e $d_a = z_{x,a} - z_{y,a}$. **Adaptação**: Porto utilizou apenas atributos ordenáveis; aqui os nominais (curso, ocupação etc.) não possuem ordem, então $d_a = 0$ se iguais e $1$ caso contrário.

**Q2.1** *(AT2.04)* Bibliotecas adicionais e funções de similaridade (vetorizadas com NumPy; cada matriz tem $N \times N$).

In [16]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

dfPrep = pd.read_csv(os.path.join(DIR_RES, "dados_preparados.csv"))
dicionario = pd.read_csv(os.path.join(DIR_RES, "dicionario_dados.csv"), index_col=0)
atribSel = list(dicionario.index)
N = len(dfPrep)
X = dfPrep[atribSel].to_numpy(dtype=np.int16)


def similaridade_m1(X):
    """M1 – Contagem Direta: fração de atributos iguais entre cada par."""
    n, p = X.shape
    S = np.zeros((n, n), dtype=np.float32)
    for a in range(p):
        S += (X[:, a][:, None] == X[:, a][None, :])
    return S / p


def similaridade_m2(df, atributos, grupos, pesos):
    """M2 – Multirelacional (Porto, 2022): coincidências por grupo relativizadas
    pelo total de coincidências de cada estudante (P_G,x + P_G,y), ponderadas por grupo."""
    n = len(df)
    R = np.zeros((n, n), dtype=np.float32)
    for g, peso in pesos.items():
        colunas = [a for a in atributos if grupos[a] == g]
        E = np.zeros((n, n), dtype=np.float32)   # coincidências do par no grupo
        P = np.zeros(n, dtype=np.float32)        # coincidências de cada nó com todos
        for a in colunas:
            x = df[a].to_numpy()
            E += (x[:, None] == x[None, :])
            P += pd.Series(x).map(pd.Series(x).value_counts()).to_numpy() - 1
        R += peso * E / (P[:, None] + P[None, :])
    np.fill_diagonal(R, 0)
    return R / R.max()                           # normaliza para [0, 1]


def similaridade_m3(df, atributos, tipos, estados):
    """M3 – Medida Ordinal (Porto, 2022): 1 - distância Euclidiana normalizada.
    Ordinais/binários: z = (r-1)/(A-1). Nominais: 0 se iguais, 1 se diferentes."""
    ordinais = [a for a in atributos if tipos[a] != "nominal"]
    nominais_ = [a for a in atributos if tipos[a] == "nominal"]
    A = np.array([estados[a] for a in ordinais])
    Z = ((df[ordinais].to_numpy() - 1) / np.maximum(A - 1, 1)).astype(np.float32)
    quad = (Z ** 2).sum(axis=1)
    D2 = quad[:, None] + quad[None, :] - 2 * (Z @ Z.T)     # soma dos (z_x - z_y)^2
    for a in nominais_:
        x = df[a].to_numpy()
        D2 += (x[:, None] != x[None, :])
    np.maximum(D2, 0, out=D2)                              # remove resíduos numéricos negativos
    return 1 - np.sqrt(D2 / len(atributos))

def fmt(n):
    """Formata inteiros com ponto como separador de milhar (ex.: 9.783.676)."""
    return f"{int(n):,}".replace(",", ".")

print(f"N = {N} nós | {len(atribSel)} atributos | pares possíveis = {fmt(N*(N-1)//2)}")

N = 4424 nós | 30 atributos | pares possíveis = 9.783.676


**Q2.2** *(AT2.04, AT2.08)* Cálculo das matrizes de similaridade. Para economizar memória, guarda-se apenas o triângulo superior (cada par uma vez, sem *self-loops*).

In [17]:
PESOS_M2 = {"DM": 1, "SE": 2, "AC": 3}       # a priori (Kim et al., 2023; Jiménez-Macías et al., 2023)

grupoSel = dicionario["grupo"].to_dict()
tipoSel = dicionario["tipo"].to_dict()
estadosSel = dicionario["estados_final"].to_dict()

iu = np.triu_indices(N, k=1)                   # pares (i < j): exclui self-loops
origem, destino = iu[0].astype(np.int32), iu[1].astype(np.int32)
del iu

construtores = {
    "M1": lambda: similaridade_m1(X),
    "M2": lambda: similaridade_m2(dfPrep, atribSel, grupoSel, PESOS_M2),
    "M3": lambda: similaridade_m3(dfPrep, atribSel, tipoSel, estadosSel),
}
sim = {}        # modelo -> vetor de similaridades dos pares (triângulo superior)
tempoSim = {}
for nome, construir in construtores.items():
    t0 = time.time()
    S = construir()
    sim[nome] = S[origem, destino].astype(np.float32)
    tempoSim[nome] = time.time() - t0
    del S
    print(f"{nome:10s}: {tempoSim[nome]:.2f} s | média = {sim[nome].mean():.3f} | "
          f"mín = {sim[nome].min():.3f} | máx = {sim[nome].max():.3f}")

M1        : 0.58 s | média = 0.407 | mín = 0.033 | máx = 0.967


M2        : 1.25 s | média = 0.228 | mín = 0.015 | máx = 1.000


M3        : 0.58 s | média = 0.431 | mín = 0.097 | máx = 0.974


**Q2.3** *(AT2.05)* Distribuição dos pesos (similaridades) em cada modelo. Indica em que faixa de W a rede deixa de ser quase completa e começa a se fragmentar.

In [18]:
MODELOS = ["M1", "M2", "M3"]
CORES_MODELO = {"M1": "#4c72b0", "M2": "#dd8452", "M3": "#55a868"}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
amostra = np.random.default_rng(SEED).choice(len(origem), size=500_000, replace=False)
for ax, m in zip(axes, MODELOS):
    ax.hist(sim[m][amostra], bins=60, color=CORES_MODELO[m], edgecolor="white")
    q = np.percentile(sim[m], [50, 99])
    ax.axvline(q[0], color="black", ls="--", lw=1, label=f"mediana = {q[0]:.2f}")
    ax.axvline(q[1], color="gray", ls=":", lw=1, label=f"p99 = {q[1]:.2f}")
    ax.set_title(f"{m}"); ax.set_xlabel("similaridade (peso da aresta)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("pares (amostra de 500 mil)")
fig.suptitle("Distribuição das similaridades entre pares de estudantes", fontsize=12)
plt.tight_layout()
salvar_fig("2_3_distribuicao_pesos")
plt.show()

Figura salva: figuras/2_3_distribuicao_pesos.png


**Q2.4** *(AT2.08)* Funções para construir a rede para um limiar W, calcular estatísticas estruturais e detectar comunidades.

* Estatísticas baratas (L, densidade, ⟨k⟩, isolados, componentes, GCC) são calculadas para **todos** os W.
* Detecção de comunidades só é aplicada quando o número de arestas é viável para cada algoritmo (redes quase completas, além de caras, não têm estrutura de comunidade interpretável). Como em Porto (2022), **nós isolados contam como comunidades unitárias**.

In [19]:
LIMITE_ARESTAS = {            # máximo de arestas para executar cada algoritmo
    "Louvain": 6_000_000,     # O(|E|) na prática
    "Fast Greedy": 2_500_000, # O(|E| d log|V|)
    "Walktrap": 600_000,      # O(|E||V|^2) no pior caso
}


def estatisticas_rede(s, w):
    """Estatísticas estruturais da rede com arestas s >= w (sem construir o igraph)."""
    masc = s >= w
    L = int(masc.sum())
    graus = np.bincount(origem[masc], minlength=N) + np.bincount(destino[masc], minlength=N)
    adj = coo_matrix((np.ones(L, dtype=np.int8), (origem[masc], destino[masc])), shape=(N, N))
    nComp, rotulos = connected_components(adj, directed=False)
    gcc = np.bincount(rotulos).max()
    return {"L": L, "densidade": L / (N * (N - 1) / 2), "k_medio": 2 * L / N,
            "isolados": int((graus == 0).sum()), "componentes": int(nComp),
            "gcc_frac": gcc / N}


def construir_rede(s, w):
    """Grafo igraph não direcionado e ponderado com as arestas de similaridade >= w."""
    masc = s >= w
    g = ig.Graph(n=N, edges=np.column_stack((origem[masc], destino[masc])), directed=False)
    g.es["weight"] = s[masc].astype(float)
    return g


def detectar_comunidades(g, metodo):
    """Executa o método e devolve a partição (igraph.VertexClustering)."""
    random.seed(SEED)                                    # igraph usa o gerador do módulo random
    if metodo == "Louvain":
        return g.community_multilevel(weights="weight")
    if metodo == "Fast Greedy":
        return g.community_fastgreedy(weights="weight").as_clustering()
    if metodo == "Walktrap":
        return g.community_walktrap(weights="weight").as_clustering()
    raise ValueError(metodo)


def resumo_particao(g, part):
    """#C, modularidade, arestas entre grupos (#AG) e menor comunidade (% de N)."""
    memb = np.array(part.membership)
    ext = g.get_edgelist()
    arr = np.array(ext) if len(ext) else np.zeros((0, 2), dtype=int)
    ag = int((memb[arr[:, 0]] != memb[arr[:, 1]]).sum()) if len(arr) else 0
    tam = np.bincount(memb)
    return {"C": len(tam), "M": g.modularity(memb, weights="weight"), "AG": ag,
            "menor_com_%": 100 * tam.min() / N, "maior_com_%": 100 * tam.max() / N}

print("Funções da seção 2 carregadas.")

Funções da seção 2 carregadas.


**Q2.5** *(AT2.08, AT2.10)* Varredura do limiar **W de 0% a 90%, em passos de 5%** (Tabela 3 de Porto, 2022), para os três modelos e os três métodos (Fast Greedy, Louvain/Multilevel e Walktrap).

⏱️ *Esta é a célula mais demorada (estimativa: 8 a 15 min, conforme o computador). O progresso é exibido a cada W.*

In [20]:
LIMIARES = np.round(np.arange(0.0, 0.91, 0.05), 2)
METODOS = ["Fast Greedy", "Louvain", "Walktrap"]

registros = []
t0Total = time.time()
for m in MODELOS:
    metodosModelo = METODOS
    for w in LIMIARES:
        est = estatisticas_rede(sim[m], w)
        base = {"modelo": m, "w": w, **est}
        aplicaveis = [mt for mt in metodosModelo if 0 < est["L"] <= LIMITE_ARESTAS[mt]]
        g = construir_rede(sim[m], w) if aplicaveis else None
        for mt in metodosModelo:
            linha = {**base, "metodo": mt}
            if mt in aplicaveis:
                t0 = time.time()
                part = detectar_comunidades(g, mt)
                linha.update(resumo_particao(g, part))
                linha["tempo_s"] = time.time() - t0
            registros.append(linha)
        print(f"{m:10s} W={w:.2f} | L={fmt(est['L']):>9} | GCC={est['gcc_frac']:.3f} | "
              f"isolados={est['isolados']:>4} | métodos: {', '.join(aplicaveis) or '—'}")
        del g

varredura = pd.DataFrame(registros)
varredura.to_csv(os.path.join(DIR_RES, "varredura_w.csv"), index=False)
print(f"\nTempo total da varredura: {(time.time() - t0Total)/60:.1f} min")

M1         W=0.00 | L=9.783.676 | GCC=1.000 | isolados=   0 | métodos: —


M1         W=0.05 | L=9.783.634 | GCC=1.000 | isolados=   0 | métodos: —


M1         W=0.10 | L=9.783.186 | GCC=1.000 | isolados=   0 | métodos: —


M1         W=0.15 | L=9.765.172 | GCC=1.000 | isolados=   0 | métodos: —


M1         W=0.20 | L=9.711.727 | GCC=1.000 | isolados=   0 | métodos: —


...

M3         W=0.50 | L=1.797.150 | GCC=1.000 | isolados=   0 | métodos: Fast Greedy, Louvain


M3         W=0.55 | L=  623.455 | GCC=0.998 | isolados=   9 | métodos: Fast Greedy, Louvain


M3         W=0.60 | L=  171.622 | GCC=0.960 | isolados= 165 | métodos: Fast Greedy, Louvain, Walktrap


M3         W=0.65 | L=   39.640 | GCC=0.807 | isolados= 714 | métodos: Fast Greedy, Louvain, Walktrap


M3         W=0.70 | L=    8.576 | GCC=0.456 | isolados=1933 | métodos: Fast Greedy, Louvain, Walktrap
M3         W=0.75 | L=    1.568 | GCC=0.114 | isolados=3243 | métodos: Fast Greedy, Louvain, Walktrap
M3         W=0.80 | L=      275 | GCC=0.005 | isolados=4045 | métodos: Fast Greedy, Louvain, Walktrap
M3         W=0.85 | L=       90 | GCC=0.001 | isolados=4261 | métodos: Fast Greedy, Louvain, Walktrap
M3         W=0.90 | L=       28 | GCC=0.001 | isolados=4371 | métodos: Fast Greedy, Louvain, Walktrap

Tempo total da varredura: 3.1 min


**Q2.6** *(AT2.10)* Tabela no formato da Tabela 5 de Porto (2022): para cada W, arestas entre grupos (#AG), modularidade (M) e quantidade de comunidades (#C). Células vazias = método não executado (rede densa demais).

In [21]:
tab5 = (varredura
        .pivot_table(index="w", columns=["modelo", "metodo"], values=["AG", "M", "C"])
        .reorder_levels([1, 2, 0], axis=1).sort_index(axis=1))
tab5.index = [f"{int(w*100)}%" for w in tab5.index]
tab5.to_csv(os.path.join(DIR_RES, "tabela_porto_formato.csv"))

with pd.option_context("display.max_columns", None, "display.width", 250):
    for m in MODELOS:
        print(f"\n=== {m} ===")
        t = tab5[m].copy()
        for mt in METODOS:
            if mt in t.columns.get_level_values(0):
                t[(mt, "M")] = t[(mt, "M")].round(3)
        print(t.to_string(na_rep="—"))


=== M1 ===
metodo Fast Greedy                   Louvain                Walktrap               
                AG       C      M         AG       C      M       AG       C      M
25%              —       —      —          —       —      —        —       —      —
30%              —       —      —          —       —      —        —       —      —
35%              —       —      —          —       —      —        —       —      —
40%              —       —      —  3094327.0     4.0  0.135        —       —      —
45%              —       —      —  1277063.0     4.0  0.258        —       —      —
50%       669550.0     3.0  0.317   676250.0     4.0  0.340        —       —      —
55%       124247.0    10.0  0.499   164498.0    11.0  0.511        —       —      —
60%        42363.0    34.0  0.560    62581.0    31.0  0.581  60776.0    94.0  0.575
65%         4736.0   233.0  0.652     5580.0   229.0  0.657   7168.0   360.0  0.657
70%         1037.0  1142.0  0.682     1657.0  1133.0  0.729   30

**Q2.7** *(AT2.10)* Comportamento estrutural das redes em função de W.

In [22]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
estrut = varredura.drop_duplicates(["modelo", "w"])
for m in MODELOS:
    d = estrut[estrut["modelo"] == m]
    estilo = dict(color=CORES_MODELO[m], marker="o", ms=3, ls="-", label=m)
    axes[0, 0].plot(d["w"] * 100, d["L"].replace(0, np.nan), **estilo)
    axes[0, 1].plot(d["w"] * 100, d["gcc_frac"], **estilo)
    lv = varredura[(varredura["modelo"] == m) & (varredura["metodo"] == "Louvain")]
    axes[1, 0].plot(lv["w"] * 100, lv["C"], **estilo)
    axes[1, 1].plot(lv["w"] * 100, lv["M"], **estilo)

axes[0, 0].set_yscale("log"); axes[0, 0].set_ylabel("Arestas L (log)")
axes[0, 1].set_ylabel("Fração de nós no GCC")
axes[1, 0].set_yscale("log"); axes[1, 0].set_ylabel("Nº de comunidades (Louvain, log)")
axes[1, 0].axhspan(2, 5, color="gold", alpha=0.25, label="critério: 2 a 5")
axes[1, 1].set_ylabel("Modularidade (Louvain)")
for ax in axes.ravel():
    ax.set_xlabel("Limiar W (%)"); ax.legend(fontsize=8)
fig.suptitle("Efeito do limiar de similaridade W na estrutura das redes", fontsize=13)
plt.tight_layout()
salvar_fig("2_7_varredura_w")
plt.show()

Figura salva: figuras/2_7_varredura_w.png


**Q2.8** *(AT2.10, AT2.14)* Seleção da rede baseline pelos critérios de Porto (2022, Tabela 4), aplicados em ordem de prioridade:

| Prioridade | Critério |
|---|---|
| 1 | Quantidade de comunidades entre 2 e 5 |
| 2 | Menor comunidade com ≥ 5% dos nós |
| 3 | Maior modularidade |
| 4 | Menor quantidade de arestas entre comunidades |
| 5 | Maior limiar de similaridade W (desempate) |

In [23]:
cand = varredura[varredura["modelo"].isin(MODELOS) & varredura["C"].notna()].copy()
cand = cand[(cand["C"] >= 2) & (cand["C"] <= 5) & (cand["menor_com_%"] >= 5)]
cand = cand.sort_values(["M", "AG", "w"], ascending=[False, True, False])

colunasVer = ["modelo", "metodo", "w", "C", "M", "AG", "L", "gcc_frac", "menor_com_%", "maior_com_%"]
print(f"Redes que atendem aos critérios 1 e 2: {len(cand)}")
print("\nMelhor rede por modelo:")
print(cand.groupby("modelo").head(1)[colunasVer].round(3).to_string(index=False))
print("\nTop 10 geral:")
print(cand[colunasVer].head(10).round(3).to_string(index=False))

escolha = cand.iloc[0]
MODELO_BASE, METODO_BASE, W_BASE = escolha["modelo"], escolha["metodo"], float(escolha["w"])
print(f"\n>>> Rede baseline: {MODELO_BASE} | {METODO_BASE} | W = {W_BASE:.0%} | "
      f"C = {int(escolha['C'])} | M = {escolha['M']:.3f}")

Redes que atendem aos critérios 1 e 2: 10

Melhor rede por modelo:
modelo      metodo    w   C     M       AG       L  gcc_frac  menor_com_%  maior_com_%
    M2 Fast Greedy 0.35 4.0 0.577  22252.0  178048       1.0       13.675       36.008
    M1     Louvain 0.50 4.0 0.340 676250.0 2166128       1.0        7.798       37.477
    M3     Louvain 0.50 4.0 0.220 957601.0 1797150       1.0       17.202       39.399

Top 10 geral:
modelo      metodo    w   C     M        AG       L  gcc_frac  menor_com_%  maior_com_%
    M2 Fast Greedy 0.35 4.0 0.577   22252.0  178048       1.0       13.675       36.008
    M2     Louvain 0.30 5.0 0.523  210346.0  844565       1.0       12.138       31.962
    M2 Fast Greedy 0.30 3.0 0.453  146028.0  844565       1.0       14.693       50.701
    M1     Louvain 0.50 4.0 0.340  676250.0 2166128       1.0        7.798       37.477
    M1 Fast Greedy 0.50 3.0 0.317  669550.0 2166128       1.0       15.574       47.581
    M2     Louvain 0.25 4.0 0.288 1195643.

**Q2.9** *(AT2.10)* Construção e exportação da rede baseline (com atributos dos nós e a comunidade de cada estudante) para as seções 3–5 e para o Gephi.

In [24]:
gBase = construir_rede(sim[MODELO_BASE], W_BASE)
partBase = detectar_comunidades(gBase, METODO_BASE)

gBase.vs["id"] = dfPrep["id"].tolist()
gBase.vs["target"] = dfPrep["target"].tolist()
gBase.vs["comunidade"] = [f"C{c+1}" for c in partBase.membership]
for a in atribSel:
    gBase.vs[a] = dfPrep[a].tolist()

gBase.write_pickle(os.path.join(DIR_RES, "rede_baseline.pkl"))
gBase.write_graphml(os.path.join(DIR_RES, "rede_baseline.graphml"))   # abrir no Gephi

print(f"Rede baseline ({MODELO_BASE}, W = {W_BASE:.0%}, {METODO_BASE})")
print(f"  N = {gBase.vcount()} | L = {fmt(gBase.ecount())} | densidade = {gBase.density():.4f}")
print(f"  Comunidades = {len(partBase)} | tamanhos = {sorted(partBase.sizes(), reverse=True)}")
print(f"  Modularidade = {partBase.modularity:.3f}")
print("Arquivos salvos: rede_baseline.pkl e rede_baseline.graphml")

Rede baseline (M2, W = 35%, Fast Greedy)
  N = 4424 | L = 178.048 | densidade = 0.0182
  Comunidades = 4 | tamanhos = [1593, 1416, 810, 605]
  Modularidade = 0.577
Arquivos salvos: rede_baseline.pkl e rede_baseline.graphml


---
## **3. Caracterização da Rede Baseline e Comunidades** (CRISP-NET Fase 4 / AT2 + H1)

Análise da rede escolhida na seção 2. Distâncias (caminho médio, diâmetro, betweenness, closeness) são calculadas **sem pesos** (número de saltos), pois o peso representa similaridade e não custo; a modularidade usa os pesos.

**Q3.1** *(AT2.11, AT2.12, AT3.01)* Métricas globais da rede baseline (tabela do artigo e referência para a AT3).

In [25]:
gBase = ig.Graph.Read_Pickle(os.path.join(DIR_RES, "rede_baseline.pkl"))
membBase = np.array([int(c[1:]) - 1 for c in gBase.vs["comunidade"]])
grausBase = np.array(gBase.degree())
gccBase = gBase.connected_components().giant()

metricasBase = pd.Series({
    "Ordem N": gBase.vcount(),
    "Tamanho L": gBase.ecount(),
    "Densidade": gBase.density(),
    "Grau médio <k>": grausBase.mean(),
    "Grau mínimo": grausBase.min(),
    "Grau máximo": grausBase.max(),
    "Heterogeneidade <k²>/<k>": (grausBase ** 2).mean() / grausBase.mean(),
    "Componentes conexos": len(gBase.connected_components()),
    "GCC (% dos nós)": 100 * gccBase.vcount() / gBase.vcount(),
    "Nós isolados": int((grausBase == 0).sum()),
    "Caminho médio <d> (GCC)": gccBase.average_path_length(),
    "Diâmetro (GCC)": gccBase.diameter(),
    "Clustering médio (local)": gBase.transitivity_avglocal_undirected(mode="zero"),
    "Clustering global (transitividade)": gBase.transitivity_undirected(),
    "Assortatividade de grau": gBase.assortativity_degree(directed=False),
    "Modularidade (partição baseline)": gBase.modularity(membBase, weights="weight"),
}, name="valor")
metricasBase.to_csv(os.path.join(DIR_RES, "metricas_baseline.csv"))
metricasBase.round(4).to_frame()

,valor
Ordem N,4424.0000
Tamanho L,178048.0000
Densidade,0.0182
Grau médio <k>,80.4919
Grau mínimo,3.0000
Grau máximo,349.0000
Heterogeneidade <k²>/<k>,129.4159
Componentes conexos,1.0000
GCC (% dos nós),100.0000
Nós isolados,0.0000


**Q3.2** *(AT2.11, AT3.07)* Distribuição de graus: escala linear e log-log, com a Poisson de mesmo ⟨k⟩ (referência de rede aleatória).

In [26]:
valoresK, contagemK = np.unique(grausBase, return_counts=True)
pk = contagemK / contagemK.sum()
kMedio = grausBase.mean()
pkPoisson = np.exp(-kMedio + valoresK * np.log(kMedio) - np.array([math.lgamma(k + 1) for k in valoresK]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(grausBase, bins=60, color="#4c72b0", edgecolor="white")
axes[0].axvline(kMedio, color="black", ls="--", label=f"<k> = {kMedio:.1f}")
axes[0].set_xlabel("Grau k"); axes[0].set_ylabel("Nº de nós"); axes[0].legend()
axes[0].set_title("Distribuição de graus (linear)")
axes[1].loglog(valoresK, pk, "o", ms=3, alpha=0.6, label="Empírica")
axes[1].loglog(valoresK, pkPoisson, "-", color="#d62728", label="Poisson (mesmo <k>)")
axes[1].set_ylim(pk.min() / 2, pk.max() * 2)
axes[1].set_xlabel("Grau k (log)"); axes[1].set_ylabel("P(k) (log)"); axes[1].legend()
axes[1].set_title("Distribuição de graus (log-log)")
plt.tight_layout()
salvar_fig("3_2_distribuicao_graus")
plt.show()

Figura salva: figuras/3_2_distribuicao_graus.png


**Q3.3** *(AT2.12, AT4.10)* Centralidades (grau, betweenness, closeness, autovetor) e nós mais centrais.

In [27]:
t0 = time.time()
dfNos = pd.DataFrame({
    "id": gBase.vs["id"],
    "target": gBase.vs["target"],
    "comunidade": gBase.vs["comunidade"],
    "grau": grausBase,
    "betweenness": np.array(gBase.betweenness(directed=False)) / ((N - 1) * (N - 2) / 2),
    "closeness": gBase.closeness(),
    "autovetor": gBase.eigenvector_centrality(),
})
dfNos.to_csv(os.path.join(DIR_RES, "centralidades_nos.csv"), index=False)
print(f"Centralidades calculadas em {time.time() - t0:.1f} s")

centr = ["grau", "betweenness", "closeness", "autovetor"]
print("\nCorrelação de Spearman entre centralidades:")
print(dfNos[centr].corr(method="spearman").round(3).to_string())

print("\nCentralidade média por Target:")
print(dfNos.groupby("target")[centr].mean().round(4).to_string())

topGrau = dfNos.nlargest(10, "grau")[["id", "grau", "betweenness", "comunidade", "target"]]
topBet = dfNos.nlargest(10, "betweenness")[["id", "betweenness", "grau", "comunidade", "target"]]
print("\nTop 10 por grau:");        print(topGrau.round(5).to_string(index=False))
print("\nTop 10 por betweenness:"); print(topBet.round(5).to_string(index=False))

Centralidades calculadas em 13.3 s

Correlação de Spearman entre centralidades:
              grau  betweenness  closeness  autovetor
grau         1.000        0.555      0.811      0.700
betweenness  0.555        1.000      0.790      0.342
closeness    0.811        0.790      1.000      0.601
autovetor    0.700        0.342      0.601      1.000

Centralidade média por Target:
             grau  betweenness  closeness  autovetor
target                                              
Dropout   89.0605       0.0005     0.3469     0.1883
Enrolled  51.7746       0.0003     0.3306     0.0659
Graduate  85.3019       0.0004     0.3413     0.0654

Top 10 por grau:
  id  grau  betweenness comunidade   target
 951   349      0.00250         C3  Dropout
2748   342      0.00184         C3  Dropout
3682   317      0.00094         C3  Dropout
 711   312      0.00118         C3  Dropout
 727   298      0.00076         C3  Dropout
3536   298      0.00090         C3  Dropout
2794   297      0.00120    

**Q3.4** *(AT2.13)* Visualizações da rede:
* **(a) Visão mesoscópica** (como em Porto, 2022): cada círculo é uma comunidade (área ∝ nº de estudantes) e a espessura das ligações ∝ nº de arestas entre comunidades.
* **(b) Rede completa** (layout Fruchterman–Reingold), colorida por comunidade e por *Target*; tamanho ∝ grau. Para legibilidade, só 15 mil arestas sorteadas são desenhadas. A rede completa pode ser explorada no Gephi com `resultados/rede_baseline.graphml`.

In [28]:
nomesCom = sorted(set(gBase.vs["comunidade"]), key=lambda c: int(c[1:]))
nCom = len(nomesCom)
CORES_COM = plt.cm.Set2(np.arange(nCom))
tamCom = np.bincount(membBase)
fracDrop = [np.mean(np.array(gBase.vs["target"])[membBase == c] == "Dropout") for c in range(nCom)]

# (a) Visão mesoscópica
arestas = np.array(gBase.get_edgelist())
cu, cv = membBase[arestas[:, 0]], membBase[arestas[:, 1]]
entre = np.zeros((nCom, nCom), dtype=int)
np.add.at(entre, (np.minimum(cu, cv), np.maximum(cu, cv)), 1)

ang = np.linspace(0, 2 * np.pi, nCom, endpoint=False) + np.pi / 4
pos = np.column_stack((np.cos(ang), np.sin(ang)))
fig, ax = plt.subplots(figsize=(6.5, 6.5))
maxEntre = max(entre[i, j] for i in range(nCom) for j in range(i + 1, nCom))
for i in range(nCom):
    for j in range(i + 1, nCom):
        if entre[i, j] > 0:
            ax.plot(*zip(pos[i], pos[j]), color="gray", lw=1 + 9 * entre[i, j] / maxEntre, zorder=1)
            ax.text(*(pos[i] + 0.3 * (pos[j] - pos[i])), fmt(entre[i, j]), fontsize=8, ha="center",
                    bbox=dict(fc="white", ec="none", alpha=0.8), zorder=3)
for c in range(nCom):
    ax.scatter(*pos[c], s=tamCom[c] * 3, color=CORES_COM[c], edgecolor="black", zorder=2)
    ax.text(*pos[c], f"{nomesCom[c]}\n({tamCom[c]})\nDropout {100*fracDrop[c]:.0f}%",
            ha="center", va="center", fontsize=9, fontweight="bold", zorder=4)
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6); ax.axis("off")
ax.set_title(f"Visão mesoscópica — arestas internas: {fmt(np.trace(entre))} | entre comunidades: "
             f"{fmt(entre.sum() - np.trace(entre))}", fontsize=10)
salvar_fig("3_4a_visao_mesoscopica")
plt.show()

# (b) Rede completa: layout Fruchterman–Reingold (todos os nós; 15 mil arestas sorteadas apenas para o desenho)
random.seed(SEED)
lay = np.array(gBase.layout_fruchterman_reingold(niter=500, grid=True).coords)
rng = np.random.default_rng(SEED)
arDesenho = arestas[rng.choice(len(arestas), size=min(15_000, len(arestas)), replace=False)]
targBase = np.array(gBase.vs["target"])
tamNo = 2 + 25 * grausBase / grausBase.max()

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, cores, titulo in [
        (axes[0], [CORES_COM[c] for c in membBase], "Cor = comunidade"),
        (axes[1], [CORES_TARGET[t] for t in targBase], "Cor = Target")]:
    ax.plot(lay[arDesenho].transpose(1, 2, 0)[:, 0], lay[arDesenho].transpose(1, 2, 0)[:, 1],
            color="gray", lw=0.2, alpha=0.05, zorder=1)
    ax.scatter(lay[:, 0], lay[:, 1], s=tamNo, c=cores, edgecolor="none", alpha=0.8, zorder=2)
    ax.set_title(titulo); ax.axis("off")
for c in range(nCom):
    axes[0].scatter([], [], color=CORES_COM[c], label=nomesCom[c])
for t, cor in CORES_TARGET.items():
    axes[1].scatter([], [], color=cor, label=t)
axes[0].legend(loc="lower left"); axes[1].legend(loc="lower left")
fig.suptitle(f"Rede baseline ({fmt(N)} nós; {fmt(len(arDesenho))} de {fmt(len(arestas))} arestas desenhadas), tamanho ∝ grau", fontsize=12)
plt.tight_layout()
salvar_fig("3_4b_rede_completa")
plt.show()

Figura salva: figuras/3_4a_visao_mesoscopica.png


Figura salva: figuras/3_4b_rede_completa.png


**Q3.5** *(RQ1/H1)* Perfil das comunidades: composição por *Target* e características médias dos estudantes (dados originais, antes da discretização).

In [29]:
dfCom = df.copy()
dfCom["comunidade"] = gBase.vs["comunidade"]
fracGlobalDrop = (df["target"] == "Dropout").mean()

perfil = pd.crosstab(dfCom["comunidade"], dfCom["target"], normalize="index").mul(100)
perfil.insert(0, "n", dfCom["comunidade"].value_counts())
perfil["lift_Dropout"] = perfil["Dropout"] / (100 * fracGlobalDrop)
caracteristicas = dfCom.groupby("comunidade").agg(
    idade=("idade_ingresso", "mean"),
    nota_admissao=("nota_admissao", "mean"),
    uc1_aprovadas=("uc1_aprovadas", "mean"),
    uc2_nota=("uc2_nota", "mean"),
    devedor_pct=("devedor", lambda s: 100 * s.mean()),
    mensalidade_em_dia_pct=("mensalidade_em_dia", lambda s: 100 * s.mean()),
    bolsista_pct=("bolsista", lambda s: 100 * s.mean()),
    deslocado_pct=("deslocado", lambda s: 100 * s.mean()),
    masculino_pct=("genero", lambda s: 100 * s.mean()),
    noturno_pct=("turno", lambda s: 100 * (1 - s.mean())),
)
perfil = perfil.join(caracteristicas).loc[nomesCom]
perfil.to_csv(os.path.join(DIR_RES, "perfil_comunidades.csv"))

# Composição por Target (barras empilhadas)
fig, ax = plt.subplots(figsize=(7, 3.8))
base = np.zeros(nCom)
for t in ["Dropout", "Enrolled", "Graduate"]:
    ax.barh(nomesCom, perfil[t], left=base, color=CORES_TARGET[t], label=t)
    for i, v in enumerate(perfil[t]):
        if v > 6:
            ax.text(base[i] + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white")
    base += perfil[t].values
ax.axvline(100 * fracGlobalDrop, color="black", ls="--", lw=1, label=f"Dropout global ({100*fracGlobalDrop:.0f}%)")
ax.set_xlabel("% dos estudantes da comunidade"); ax.invert_yaxis()
ax.legend(fontsize=8, loc="lower right", bbox_to_anchor=(1, 1.02), ncol=4)
plt.tight_layout()
salvar_fig("3_5_composicao_comunidades")
plt.show()

perfil.round(2).T

Figura salva: figuras/3_5_composicao_comunidades.png


comunidade,C1,C2,C3,C4
n,810.00,1593.00,1416.00,605.00
Dropout,33.46,17.07,58.83,7.44
Enrolled,26.05,20.15,13.42,11.90
Graduate,40.49,62.77,27.75,80.66
lift_Dropout,1.04,0.53,1.83,0.23
idade,19.80,20.10,30.26,19.86
nota_admissao,127.50,127.35,126.19,127.14
uc1_aprovadas,2.56,5.26,4.46,6.71
uc2_nota,7.95,12.22,8.03,13.18
devedor_pct,5.80,5.84,25.14,1.16


**Q3.6** *(H1)* Testes da hipótese H1 — *estudantes semelhantes formam comunidades densas e a evasão não se distribui ao acaso entre elas*:

| Critério (definido na AT1) | Teste |
|---|---|
| (a) Modularidade acima do modelo nulo | Q empírica × 30 redes religadas preservando os graus (*configuration model*); z > 2 |
| (b) Homofilia da evasão | Assortatividade nominal do *Target*; teste de permutação (1.000 embaralhamentos), p < 0,05 |
| (c) Concentração de evadidos | Alguma comunidade com fração de Dropout ≥ 1,5 × a global; associação comunidade × Target (χ², V de Cramér) |

In [30]:
N_REPLICAS = 30
N_PERMUT = 1000

# (a) Modularidade empírica × modelo nulo (Louvain sem pesos nas duas, para comparação justa)
random.seed(SEED)
qEmp = gBase.community_multilevel().modularity
qNulo = []
for r in range(N_REPLICAS):
    random.seed(SEED + r)
    gNulo = gBase.copy()
    gNulo.rewire(n=2 * gNulo.ecount())             # troca de arestas preservando os graus
    qNulo.append(gNulo.community_multilevel().modularity)
qNulo = np.array(qNulo)
zQ = (qEmp - qNulo.mean()) / qNulo.std(ddof=1)

# (b) Assortatividade nominal do Target com teste de permutação
codTarget = pd.factorize(pd.Series(gBase.vs["target"]))[0].tolist()
rEmp = gBase.assortativity_nominal(codTarget, directed=False)
rngP = np.random.default_rng(SEED)
rPerm = np.array([gBase.assortativity_nominal(rngP.permutation(codTarget).tolist(), directed=False)
                  for _ in range(N_PERMUT)])
pAssort = (np.sum(rPerm >= rEmp) + 1) / (N_PERMUT + 1)

# (c) Concentração e associação comunidade × Target
tabCT = pd.crosstab(dfCom["comunidade"], dfCom["target"])
chi2, pChi, gl, _ = chi2_contingency(tabCT)
vCramer = cramer_v(dfCom["comunidade"], dfCom["target"])
maxLift = perfil["lift_Dropout"].max()
comMax = perfil["lift_Dropout"].idxmax()

h1 = pd.DataFrame([
    ["(a) Modularidade × nulo", f"Q = {qEmp:.3f} vs {qNulo.mean():.3f} ± {qNulo.std(ddof=1):.3f}",
     f"z = {zQ:.1f}", zQ > 2],
    ["(b) Assortatividade do Target", f"r = {rEmp:.3f} (nulo: {rPerm.mean():.3f} ± {rPerm.std():.3f})",
     f"p = {pAssort:.4f}", (rEmp > 0) and (pAssort < 0.05)],
    ["(c) Concentração de Dropout", f"maior lift = {maxLift:.2f} ({comMax})",
     f"χ² = {chi2:.0f}, p = {pChi:.1e}, V = {vCramer:.3f}", maxLift >= 1.5],
], columns=["critério", "estatística", "teste", "atendido"])
h1.to_csv(os.path.join(DIR_RES, "testes_h1.csv"), index=False)
print(h1.to_string(index=False))
print(f"\nH1 {'CONFIRMADA' if h1['atendido'].all() else 'PARCIALMENTE confirmada' if h1['atendido'].any() else 'REJEITADA'}"
      f" ({h1['atendido'].sum()}/3 critérios)")

                     critério                      estatística                             teste  atendido
      (a) Modularidade × nulo       Q = 0.668 vs 0.088 ± 0.001                         z = 475.8      True
(b) Assortatividade do Target r = 0.417 (nulo: -0.000 ± 0.002)                        p = 0.0010      True
  (c) Concentração de Dropout           maior lift = 1.83 (C3) χ² = 925, p = 1.3e-196, V = 0.323      True

H1 CONFIRMADA (3/3 critérios)
